### Setup and prerequisites

This notebook loads the trained model and creates a prediction before generating the agricultural explanation.

In [1]:
import json
import os
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b0

from dotenv import load_dotenv


def find_project_root():
    current_path = Path.cwd().resolve()
    for path in [current_path, *current_path.parents]:
        if (path / ".git").exists() or (path / "data").is_dir() and (path / "models").is_dir():
            return path
    raise FileNotFoundError(
        "Project root not found. Run this notebook from the project directory."
    )


ROOT = find_project_root()
load_dotenv(ROOT / ".env")

MODEL_PATH = ROOT / "models" / "rice_disease_model.pth"
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Trained model not found: {MODEL_PATH}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)
class_to_idx = checkpoint["class_to_idx"]
idx_to_class = {index: name for name, index in class_to_idx.items()}
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_to_idx))
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

configured_image_path = os.getenv("INPUT_IMAGE_PATH", "").strip()
if configured_image_path:
    input_image_path = Path(configured_image_path)
    if not input_image_path.is_absolute():
        input_image_path = ROOT / input_image_path
else:
    sample_images = sorted((ROOT / "data" / "raw" / "brown_spot").glob("*.jpg"))
    if not sample_images:
        raise FileNotFoundError(
            "Set INPUT_IMAGE_PATH in .env or add a .jpg sample to data/raw/brown_spot."
        )
    input_image_path = sample_images[0]

if not input_image_path.is_file():
    raise FileNotFoundError(f"Input image not found: {input_image_path}")
input_image = Image.open(input_image_path).convert("RGB")
prediction_transform = transforms.Compose([
    transforms.Resize((checkpoint["image_size"], checkpoint["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=checkpoint["mean"], std=checkpoint["std"]),
])
input_tensor = prediction_transform(input_image).unsqueeze(0).to(device)
with torch.no_grad():
    probabilities = torch.softmax(model(input_tensor), dim=1)
predicted_index = probabilities.argmax(dim=1).item()
predicted_disease = idx_to_class[predicted_index]
model_score = probabilities[0, predicted_index].item()

print("Project root:", ROOT)
print("Input image:", input_image_path)
print("Predicted disease:", predicted_disease)
print(f"Model score: {model_score:.2%}")

Project root: D:\My_Project\Analyse\AI-Paddy-Farming-Intelligence
Input image: D:\My_Project\Analyse\AI-Paddy-Farming-Intelligence\data\raw\brown_spot\brown_spot107.jpg
Predicted disease: brown_spot
Model score: 99.88%


### Load knowledge base

In [2]:
KNOWLEDGE_PATH = (
    ROOT / "knowledge_base" / "disease_information.json"
)

with open(
    KNOWLEDGE_PATH,
    "r",
    encoding="utf-8"
) as file:

    knowledge_base = json.load(file)

disease_information = knowledge_base.get(
    predicted_disease,
    {}
)

verified_information = disease_information.get(
    "verified_information",
    ""
).strip()

print("Knowledge base loaded!")
print("Predicted category:", predicted_disease)

if not verified_information:
    print(
        "Add verified agricultural information to the "
        "knowledge base before generating an explanation."
    )

Knowledge base loaded!
Predicted category: brown_spot


### google-genai

In [3]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
GEMINI_FALLBACK_MODEL = os.getenv("GEMINI_FALLBACK_MODEL", "").strip()

if GEMINI_API_KEY:
    print("Gemini configuration loaded!")
    print("Primary model:", GEMINI_MODEL)
    print("Fallback model:", GEMINI_FALLBACK_MODEL or "not set")
else:
    print("Gemini API key is not set; local guidance will be used.")


Gemini configuration loaded!
Primary model: gemini-flash-latest
Fallback model: not set


### Generate LLM explanation

In [4]:
import time
import httpx
from google import genai

if not verified_information:
    raise ValueError(
        "No verified agricultural information found. "
        "Complete the knowledge base first."
    )

response = None
quota_exhausted = False

if GEMINI_API_KEY:
    client = genai.Client(api_key=GEMINI_API_KEY)

    prompt = f"""
You are an agricultural assistant helping rice farmers in Sri Lanka.
Use everyday words, short sentences, and explain scientific terms simply.

Disease category: {predicted_disease}
Classifier score: {model_score:.4f}

Verified agricultural information:
{verified_information}

Use only the supplied information. Do not invent pesticide names, doses,
or treatments. Say that the image result is not a confirmed field diagnosis.
If information is missing, say so. Use these headings:

Disease Identification
Symptoms
Management Recommendations
Prevention
Important Note
"""

    model_candidates = list(dict.fromkeys(
        name for name in (GEMINI_MODEL, GEMINI_FALLBACK_MODEL) if name
    ))

    try:
        for model_name in model_candidates:
            for attempt in range(3):
                try:
                    response = client.models.generate_content(
                        model=model_name,
                        contents=prompt
                    )
                    print("Response generated with model:", model_name)
                    break
                except Exception as error:
                    status_code = getattr(error, "status_code", None)
                    error_text = str(error).upper()
                    quota_exhausted = (
                        status_code == 429
                        or "429" in error_text
                        or "RESOURCE_EXHAUSTED" in error_text
                    )
                    is_temporary_failure = (
                        status_code == 503
                        or "503" in error_text
                        or isinstance(error, httpx.TransportError)
                    )
                    if quota_exhausted:
                        print("Gemini quota is exhausted; using local guidance.")
                        break
                    if not is_temporary_failure:
                        raise
                    if attempt == 2:
                        break
                    wait_seconds = 2 ** attempt
                    print(f"{model_name} timed out or is unavailable; retrying in {wait_seconds}s...")
                    time.sleep(wait_seconds)

            if response is not None or quota_exhausted:
                break
    finally:
        close_fn = getattr(client, "close", None)
        if callable(close_fn):
            close_fn()
else:
    print("Gemini API key is not configured; using local guidance.")

print("AI AGRICULTURAL ADVISORY")
print("=" * 40)

if response is None:
    print("Showing verified agricultural guidance from the local knowledge base:")
    print(verified_information)
else:
    print(response.text)


gemini-flash-latest timed out or is unavailable; retrying in 1s...
Gemini quota is exhausted; using local guidance.
AI AGRICULTURAL ADVISORY
Showing verified agricultural guidance from the local knowledge base:
Brown spot is a fungus that can travel on rice seeds. It makes brown spots on leaves. The spots may be small, or round with dark edges and lighter centers. It can also damage grains and reduce the harvest. Dry weather, poor soil, and high humidity can make it worse. Use clean, certified seed and follow local advice on soil and crop care.
